# Demo - A Short Hugging Face Tour
**Day 2 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-huggingface-tour.ipynb)

**Goal:** Show how the transformers `pipeline` helper runs pre-trained models
in a few lines across several tasks.

We run three tasks with one high-level API: sentiment analysis (BERT-style
understanding), text generation (GPT-style generation), and fill-mask (an
encoder predicting a masked word). Each pipeline downloads a small model once,
then caches it.

No API key needed.

In [1]:
# Scoped install for this demo (safe to re-run; often preinstalled on Colab).
%pip install -q torch transformers

In [2]:
from transformers import pipeline

## 1. Sentiment analysis (encoder-style understanding)
Classify the sentiment of a sentence.

In [5]:
def demo_sentiment():
    """Understanding task: classify the sentiment of a sentence."""
    print("1. Sentiment analysis (encoder-style understanding)")
    classifier = pipeline("sentiment-analysis")
    for text in ["I love this course!", "This bug ruined my afternoon.", "ArithmeticError is a subclass of Exception.", "My wife broke her leg, but she is recovering well."]:
        result = classifier(text)[0]
        print(f"  {text!r} -> {result['label']} ({result['score']:.2f})")


demo_sentiment()

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


1. Sentiment analysis (encoder-style understanding)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  'I love this course!' -> POSITIVE (1.00)
  'This bug ruined my afternoon.' -> NEGATIVE (1.00)
  'ArithmeticError is a subclass of Exception.' -> NEGATIVE (0.98)
  'My wife broke her leg, but she is recovering well.' -> POSITIVE (1.00)


## 2. Text generation (decoder-style generation)
Continue a prompt with GPT-2.

In [6]:
def demo_generation():
    """Generation task: continue a prompt with GPT-2."""
    print("2. Text generation (decoder-style generation)")
    generator = pipeline("text-generation", model="gpt2")
    out = generator(
        "In the future, software developers will",
        max_new_tokens=20,
        num_return_sequences=1,
        pad_token_id=50256,
    )
    print(f"  {out[0]['generated_text']!r}")


demo_generation()

2. Text generation (decoder-style generation)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  'In the future, software developers will be able to create their own virtual environments.\n\nIf you\'re thinking, "How do I'


## 3. Fill-mask (encoder predicting a masked word)
BERT-style model fills in a masked word using both directions of context.

In [7]:
def demo_fill_mask():
    """Understanding task: BERT fills in a masked word using both directions."""
    print("3. Fill-mask (encoder predicting a masked word)")
    fill = pipeline("fill-mask", model="distilbert-base-uncased")
    for result in fill("The capital of France is [MASK].")[:3]:
        print(f"  {result['token_str']!r}  score = {result['score']:.3f}")


demo_fill_mask()

3. Fill-mask (encoder predicting a masked word)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

  'marseille'  score = 0.143
  'nantes'  score = 0.090
  'toulouse'  score = 0.088


## Expected output
- Sentiment analysis labels the first sentence POSITIVE and the second
  NEGATIVE, each with a confidence score.
- Text generation continues the prompt with a plausible sentence about
  software development.
- Fill-mask ranks `paris` as the top candidate for the masked word.

**Takeaway:** one API, many tasks. `pipeline` handles tokenizing, running, and
decoding for you.